In [ ]:

# ============================================================
#       MOVIE RECOMMENDATION SYSTEM
#       INTERNSHIP PROJECT
#       CONTENT-BASED + COLLABORATIVE + HYBRID FILTERING
# ============================================================

# 1. IMPORT LIBRARIES

import os
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, HTML
from difflib import get_close_matches
from scipy.sparse import csr_matrix

import warnings
warnings.filterwarnings("ignore")

print("=" * 60)
print("       MOVIE RECOMMENDATION SYSTEM")
print("=" * 60)

# 2. DOWNLOAD MOVIELENS DATASET

print("\nDownloading MovieLens dataset...")

url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

if not os.path.exists("ml-latest-small"):
    urllib.request.urlretrieve(url, "movies_dataset.zip")

    with zipfile.ZipFile("movies_dataset.zip", "r") as zip_ref:
        zip_ref.extractall(".")

print("Dataset downloaded successfully!")

# 3. LOAD DATASET

movies = pd.read_csv("ml-latest-small/movies.csv")
ratings = pd.read_csv("ml-latest-small/ratings.csv")
tags = pd.read_csv("ml-latest-small/tags.csv")

print("\nDataset Information")
print("-------------------")
print("Total Movies:", len(movies))
print("Total Ratings:", len(ratings))
print("Total Users:", ratings["userId"].nunique())

# 4. DATA PREPROCESSING

movies["genres"] = movies["genres"].fillna("").str.replace(
    "|", " ", regex=False
)

movie_tags = tags.groupby("movieId")["tag"].apply(
    lambda x: " ".join(x.astype(str))
).reset_index()

movies = movies.merge(movie_tags, on="movieId", how="left")
movies["tag"] = movies["tag"].fillna("")

movies["features"] = (
    movies["genres"] + " " + movies["tag"]
)

# Calculate average ratings and rating counts

movie_stats = ratings.groupby("movieId")["rating"].agg(
    ["mean", "count"]
).reset_index()

movie_stats.columns = ["movieId", "avg_rating", "rating_count"]

movies = movies.merge(movie_stats, on="movieId", how="left")

movies["avg_rating"] = movies["avg_rating"].fillna(0)
movies["rating_count"] = movies["rating_count"].fillna(0)

# 5. CONTENT-BASED FILTERING

print("\nBuilding Content-Based Recommendation Model...")

tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(movies["features"])

content_similarity = cosine_similarity(
    tfidf_matrix, dense_output=False
)

print("Content-Based Model Ready!")

# 6. COLLABORATIVE FILTERING

print("\nBuilding Collaborative Filtering Model...")

rating_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
).fillna(0)

# Item-item similarity

item_similarity = cosine_similarity(
    rating_matrix.T,
    dense_output=False
)

movie_id_to_index = {
    movie_id: index
    for index, movie_id in enumerate(rating_matrix.columns)
}

movie_index = {
    movie_id: index
    for index, movie_id in enumerate(movies["movieId"])
}

print("Collaborative Filtering Model Ready!")

# 7. MOVIE SEARCH FUNCTION

def search_movie(movie_name):

    matches = get_close_matches(
        movie_name,
        movies["title"].tolist(),
        n=5,
        cutoff=0.25
    )

    if not matches:
        print("Movie not found. Try another movie title.")
        return None

    selected = matches[0]

    if selected.lower() != movie_name.lower():
        print("Closest matching movie:", selected)

    return movies[movies["title"] == selected].iloc[0]


# 8. CONTENT-BASED RECOMMENDATION

def content_recommend(movie_name, n=10):

    movie = search_movie(movie_name)

    if movie is None:
        return None

    idx = movie_index[movie["movieId"]]

    scores = content_similarity[idx].toarray().flatten()

    scores[idx] = -1

    top_indices = np.argsort(scores)[::-1][:n]

    result = movies.iloc[top_indices][
        ["title", "genres", "avg_rating"]
    ].copy()

    result["similarity"] = scores[top_indices]

    return result.reset_index(drop=True)


# 9. COLLABORATIVE RECOMMENDATION

def collaborative_recommend(user_id, n=10):

    if user_id not in rating_matrix.index:
        print("User ID not found!")
        return None

    user_ratings = rating_matrix.loc[user_id].values

    scores = np.asarray(
        item_similarity.T.dot(user_ratings)
    ).flatten()

    denominator = np.asarray(
        item_similarity.T.dot(
            (user_ratings > 0).astype(float)
        )
    ).flatten()

    scores = scores / np.maximum(denominator, 1e-10)

    already_rated = user_ratings > 0
    scores[already_rated] = -1

    top_indices = np.argsort(scores)[::-1][:n]

    recommended_ids = rating_matrix.columns[top_indices]

    result = movies[
        movies["movieId"].isin(recommended_ids)
    ].copy()

    score_dict = dict(zip(recommended_ids, scores[top_indices]))

    result["recommendation_score"] = result["movieId"].map(
        score_dict
    )

    result = result.sort_values(
        "recommendation_score", ascending=False
    )

    return result[
        ["title", "genres", "avg_rating", "recommendation_score"]
    ].head(n).reset_index(drop=True)


# 10. HYBRID RECOMMENDATION

def hybrid_recommend(movie_name, n=10):

    movie = search_movie(movie_name)

    if movie is None:
        return None

    idx = movie_index[movie["movieId"]]

    content_scores = content_similarity[idx].toarray().flatten()

    collab_scores = np.zeros(len(movies))

    selected_id = movie["movieId"]

    if selected_id in movie_id_to_index:

        item_idx = movie_id_to_index[selected_id]

        collab_row = item_similarity[item_idx].toarray().flatten()

        collab_dict = {
            movie_id: collab_row[i]
            for i, movie_id in enumerate(rating_matrix.columns)
        }

        for i, mid in enumerate(movies["movieId"]):
            collab_scores[i] = collab_dict.get(mid, 0)

    # Normalize scores

    if content_scores.max() > 0:
        content_scores = content_scores / content_scores.max()

    if collab_scores.max() > 0:
        collab_scores = collab_scores / collab_scores.max()

    # Combine both recommendation methods

    final_scores = (
        0.60 * content_scores +
        0.40 * collab_scores
    )

    final_scores[idx] = -1

    top_indices = np.argsort(final_scores)[::-1][:n]

    result = movies.iloc[top_indices][
        ["title", "genres", "avg_rating"]
    ].copy()

    result["hybrid_score"] = final_scores[top_indices]

    return result.reset_index(drop=True)


# 11. TOP RATED MOVIES

def top_movies(n=10):

    result = movies.sort_values(
        by=["avg_rating", "rating_count"],
        ascending=False
    )

    return result[
        ["title", "genres", "avg_rating", "rating_count"]
    ].head(n).reset_index(drop=True)


# 12. DISPLAY RECOMMENDATIONS

def display_results(data, heading):

    if data is None or data.empty:
        print("No recommendations available.")
        return

    display(HTML(
        "<h3 style='color:#2563eb'>" + heading + "</h3>"
    ))

    display(
        data.style
        .format(precision=3)
        .background_gradient(cmap="Blues")
    )


# 13. DATASET VISUALIZATION

def show_analysis():

    print("\nMOVIE DATASET ANALYSIS")

    plt.figure(figsize=(9, 4))

    top_genres = (
        movies["genres"]
        .str.split(" ")
        .explode()
    )

    top_genres = top_genres[
        top_genres != "(no genres listed)"
    ]

    top_genres.value_counts().head(10).plot(
        kind="bar"
    )

    plt.title("Top 10 Movie Genres")
    plt.xlabel("Genres")
    plt.ylabel("Number of Movies")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 4))

    ratings["rating"].value_counts().sort_index().plot(
        kind="bar"
    )

    plt.title("Movie Rating Distribution")
    plt.xlabel("Rating")
    plt.ylabel("Number of Ratings")
    plt.tight_layout()
    plt.show()


# 14. MAIN APPLICATION MENU

def movie_recommendation_system():

    while True:

        print("\n" + "=" * 55)
        print("          MOVIE RECOMMENDATION SYSTEM")
        print("=" * 55)

        print("""
        1. Content-Based Recommendations
        2. Collaborative Filtering Recommendations
        3. Hybrid Recommendations
        4. Top Rated Movies
        5. Dataset Analysis
        6. Search Movie
        7. Exit
        """)

        choice = input("Enter your choice (1-7): ").strip()

        if choice == "1":

            name = input("Enter a movie name: ")
            result = content_recommend(name)

            display_results(
                result,
                "CONTENT-BASED RECOMMENDATIONS"
            )

        elif choice == "2":

            print(
                "Available User IDs:",
                rating_matrix.index.min(),
                "to",
                rating_matrix.index.max()
            )

            try:
                uid = int(input("Enter User ID: "))
                result = collaborative_recommend(uid)

                display_results(
                    result,
                    "COLLABORATIVE FILTERING RECOMMENDATIONS"
                )

            except ValueError:
                print("Please enter a valid numeric User ID.")

        elif choice == "3":

            name = input("Enter a movie name: ")
            result = hybrid_recommend(name)

            display_results(
                result,
                "HYBRID MOVIE RECOMMENDATIONS"
            )

        elif choice == "4":

            result = top_movies()

            display_results(
                result,
                "TOP RATED MOVIES"
            )

        elif choice == "5":

            show_analysis()

        elif choice == "6":

            name = input("Enter movie title to search: ")
            movie = search_movie(name)

            if movie is not None:
                print("\nMovie Details")
                print("Title:", movie["title"])
                print("Genres:", movie["genres"])
                print("Average Rating:", movie["avg_rating"])

        elif choice == "7":

            print("\nThank you for using Movie Recommendation System!")
            break

        else:
            print("Invalid choice! Please select 1 to 7.")


# 15. RUN PROJECT

print("\nAll recommendation models are ready!")

movie_recommendation_system()


       MOVIE RECOMMENDATION SYSTEM

Dataset downloaded successfully!

Dataset Information
-------------------
Total Movies: 9742
Total Ratings: 100836
Total Users: 610

Building Content-Based Recommendation Model...
Content-Based Model Ready!

Building Collaborative Filtering Model...
Collaborative Filtering Model Ready!

All recommendation models are ready!

          MOVIE RECOMMENDATION SYSTEM

        1. Content-Based Recommendations
        2. Collaborative Filtering Recommendations
        3. Hybrid Recommendations
        4. Top Rated Movies
        5. Dataset Analysis
        6. Search Movie
        7. Exit
        
